In [ ]:
# 1. Uninstall the conflicting audio library that we don't need
!pip uninstall -y torchaudio

# 2. Cleanly install the modern Hugging Face ecosystem
!pip install -U bitsandbytes transformers datasets accelerate trl

# 3. Automatically restart the python interpreter to apply changes safely
import os
os.kill(os.getpid(), 9)

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Succes

In [ ]:
!pip install -q -U bitsandbytes datasets transformers peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

# ==========================================
# 1. SETUP MODEL AND DATASET
# ==========================================
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base model: {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load clean copy of base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load configuration matching the schema
print("Fetching dataset...")
raw_dataset = load_dataset("lavita/medical-qa-datasets", "chatdoctor-icliniq", split="test[:1000]")

# Dynamically intercept internal schema keys ('input' and 'output')
sample = raw_dataset[0]
q_key = 'question' if 'question' in sample else ('input' if 'input' in sample else list(sample.keys())[0])
a_key = 'answer_text' if 'answer_text' in sample else ('output' if 'output' in sample else list(sample.keys())[1])

def format_prompts(batch):
    formatted_texts = []
    for q, a in zip(batch[q_key], batch[a_key]):
        text = f"<|im_start|>user\nContext: You are a medical expert assistant. Answer this query: {q}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = raw_dataset.map(format_prompts, batched=True)
print("Dataset loaded and formatted successfully!")

# ==========================================
# 2. RUN EXPERIMENT 1: STANDARD LORA
# ==========================================
print("\n--- Initializing Run 1: Standard LoRA ---")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(base_model, lora_config)

# Using SFTConfig with modern 'max_length' parameter naming
training_args_lora = SFTConfig(
    output_dir="./results_lora",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer_lora = SFTTrainer(
    model=lora_model,
    train_dataset=dataset,
    args=training_args_lora
)

trainer_lora.train()
lora_model.save_pretrained("./medical_lora_adapter")
print("Standard LoRA trained and saved successfully.")

# ==========================================
# 3. RUN EXPERIMENT 2: BRAIN-INSPIRED MTLoRA
# ==========================================
print("\n--- Initializing Run 2: Brain-Inspired MTLoRA ---")

# Define structural linear transformation overlay mimicking synaptic transformation matrices
class MTLoRATransformationWrapper(nn.Module):
    def __init__(self, base_layer, rank=16):
        super().__init__()
        self.base_layer = base_layer
        # Brain-inspired scaling and rotation matrix overlay
        self.transform_matrix = nn.Parameter(torch.eye(rank, dtype=torch.bfloat16))

    def forward(self, x, *args, **kwargs):
        if hasattr(self.base_layer, 'lora_A') and 'default' in self.base_layer.lora_A:
            original_lora_A = self.base_layer.lora_A['default']
            self.base_layer.lora_A['default'].weight.data = torch.matmul(
                self.transform_matrix, original_lora_A.weight.data
            )
        return self.base_layer(x, *args, **kwargs)

# Reset model state back to baseline for an identical starting point
mtlora_model = get_peft_model(base_model, lora_config)

# Inject transformation layers into target query vectors
for name, module in mtlora_model.named_modules():
    if "q_proj" in name and hasattr(module, 'base_layer'):
        parent_name = name.rsplit('.', 1)[0]
        child_name = name.rsplit('.', 1)[-1]
        parent_module = mtlora_model.get_submodule(parent_name)
        setattr(parent_module, child_name, MTLoRATransformationWrapper(module, rank=16))

# Apply matching SFTConfig layout
training_args_mtlora = SFTConfig(
    output_dir="./results_mtlora",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer_mtlora = SFTTrainer(
    model=mtlora_model,
    train_dataset=dataset,
    args=training_args_mtlora
)

trainer_mtlora.train()
mtlora_model.save_pretrained("./medical_mtlora_adapter")
print("MTLoRA Adapter trained and saved successfully.")

Loading base model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fetching dataset...
Dataset loaded and formatted successfully!

--- Initializing Run 1: Standard LoRA ---


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Step,Training Loss
10,2.798486
20,2.452461
30,2.408504
40,2.359059
50,2.378222


Standard LoRA trained and saved successfully.

--- Initializing Run 2: Brain-Inspired MTLoRA ---


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,2.799246
20,2.453782
30,2.408261
40,2.357658
50,2.376231


MTLoRA Adapter trained and saved successfully.


In [ ]:
import os
from google.colab import files

# 1. Zip both adapter directories into a single transportable archive
print("Packaging adapters...")
!zip -r medical_adapters.zip ./medical_lora_adapter ./medical_mtlora_adapter

# 2. Trigger the browser download download hook
print("Downloading file to your local computer...")
files.download('medical_adapters.zip')
print("Download request sent! Keep this zip file handy for your Raspberry Pi.")

Packaging adapters...
  adding: medical_lora_adapter/ (stored 0%)
  adding: medical_lora_adapter/README.md (deflated 65%)
  adding: medical_lora_adapter/adapter_model.safetensors (deflated 22%)
  adding: medical_lora_adapter/adapter_config.json (deflated 58%)
  adding: medical_mtlora_adapter/ (stored 0%)
  adding: medical_mtlora_adapter/README.md (deflated 65%)
  adding: medical_mtlora_adapter/adapter_model.safetensors (deflated 22%)
  adding: medical_mtlora_adapter/adapter_config.json (deflated 58%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download request sent! Keep this zip file handy for your Raspberry Pi.


In [ ]:
# Clone the official llama.cpp repository for conversion tools
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

# Install the Python dependencies needed for the conversion scripts
!pip install -r requirements.txt
%cd ..

fatal: destination path 'llama.cpp' already exists and is not an empty directory.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
/content


In [ ]:
!pip install -U torchao>=0.16.0
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Helper function to load, merge, and export to unquantized FP16 directory
def merge_and_export(adapter_path, output_dir):
    print(f"Merging {adapter_path} with base model...")
    # Load base model in FP16 for clean merging
    base = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cpu"
    )
    # Load the adapter layers
    model = PeftModel.from_pretrained(base, adapter_path)
    # Seamlessly fuse the weights back together permanently
    merged_model = model.merge_and_unload()

    # Save the full model locally
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved merged model to {output_dir}\n")

# 1. Merge the Standard LoRA
merge_and_export("./medical_lora_adapter", "./merged_lora")

# 2. Merge the custom MTLoRA
merge_and_export("./medical_mtlora_adapter", "./merged_mtlora")

# ========================================================
# 3. CONVERT TO GGUF AND QUANTIZE (Using llama.cpp tools)
# ========================================================
print("Converting merged models to base GGUF format...")

# Convert directories to FP16 .gguf files
!python llama.cpp/convert_hf_to_gguf.py ./merged_lora --outfile ./lora_fp16.gguf
!python llama.cpp/convert_hf_to_gguf.py ./merged_mtlora --outfile ./mtlora_fp16.gguf

print("\nQuantizing models to 4-bit (Q4_K_M) for optimal Pi performance...")

# Compile the llama-quantize binary tool quickly
!make -C llama.cpp llama-quantize

# Quantize both models to lightweight versions
!./llama.cpp/llama-quantize ./lora_fp16.gguf ./medical_lora_q4.gguf Q4_K_M
!./llama.cpp/llama-quantize ./mtlora_fp16.gguf ./medical_mtlora_q4.gguf Q4_K_M

print("\nSuccess! Both models are converted to quantized GGUF format.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Merging ./medical_lora_adapter with base model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at './medical_lora_adapter'

In [ ]:
# Force install the specific Protobuf version that TensorFlow 2.20 expects
!pip install -U "protobuf>=5.28.0" "torchao>=0.16.0"

# Kill the runtime to force a clean memory reload
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have 

In [ ]:
# ========================================================
# 1. SETUP ENGINE & DEPENDENCIES VIA CMAKE
# ========================================================
print("Setting up llama.cpp tools...")
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

# Install ONLY the light conversion requirements needed for the script
!pip install -r llama.cpp/requirements/requirements-convert-hf-to-gguf.txt

# Compile using CMake instead of Makefile
print("\nBuilding llama.cpp via CMake...")
!mkdir -p llama.cpp/build
%cd llama.cpp/build
!cmake .. -DLLAMA_GGML_BACKEND=CPU
!cmake --build . --config Release --target llama-quantize
%cd ../..

print("\nExtracting uploaded adapters...")
!unzip -o medical_adapters.zip

# Create paths for the model layers
import os
os.makedirs("./merged_lora", exist_ok=True)
os.makedirs("./merged_mtlora", exist_ok=True)
!cp -r ./medical_lora_adapter/* ./merged_lora/
!cp -r ./medical_mtlora_adapter/* ./merged_mtlora/

# ========================================================
# 2. CONVERT HF DIRECTORIES TO BASE GGUF
# ========================================================
print("\nConverting directories to GGUF format...")
# Pointing to the correct script location in the modern repo structure
!python llama.cpp/convert-hf-to-gguf.py ./merged_lora --outfile ./lora_fp16.gguf --outtype f16
!python llama.cpp/convert-hf-to-gguf.py ./merged_mtlora --outfile ./mtlora_fp16.gguf --outtype f16

# ========================================================
# 3. EXPORT TO 4-BIT QUANTIZATION
# ========================================================
print("\nQuantizing models to 4-bit (Q4_K_M) for Raspberry Pi...")
# Updated path pointing to the new CMake binary output location
!./llama.cpp/build/bin/llama-quantize ./lora_fp16.gguf ./medical_lora_q4.gguf Q4_K_M
!./llama.cpp/build/bin/llama-quantize ./mtlora_fp16.gguf ./medical_mtlora_q4.gguf Q4_K_M

# Verify files exist before running the download hook
if os.path.exists('medical_lora_q4.gguf') and os.path.exists('medical_mtlora_q4.gguf'):
    print("\nSuccess! Both models are successfully built into Q4_K_M GGUF format.")
    print("Initiating browser file downloads...")
    from google.colab import files
    files.download('medical_lora_q4.gguf')
    files.download('medical_mtlora_q4.gguf')
else:
    print("\n[ERROR]: GGUF files were not created. Check terminal output log traces above.")

Setting up llama.cpp tools...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 103910, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 103910 (delta 80), reused 35 (delta 35), pack-reused 103740 (from 3)
Receiving objects: 100% (103910/103910), 409.46 MiB | 29.14 MiB/s, done.
Resolving deltas: 100% (72608/72608), done.
Updating files: 100% (3224/3224), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'llama.cpp/requirements/requirements-convert-hf-to-gguf.txt'

Building llama.cpp via CMake...
/content/llama.cpp/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI inf

In [ ]:
# ========================================================
# 1. SETUP ENGINE & COMPLETE CONVERSION SYSTEM
# ========================================================
print("Setting up llama.cpp tools...")
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!pip install gguf transformers sentencepiece numpy tokenizers peft accelerate safetensors

print("\nExtracting uploaded adapters...")
!unzip -o medical_adapters.zip

# ========================================================
# 2. CREATE A SUBPROCESS SCRIPT WITH TENSORFLOW MASKED OUT
# ========================================================
with open("merge_models.py", "w") as f:
    f.write('''
import sys
# Mask out tensorflow to completely bypass the broken protobuf import path
sys.modules['tensorflow'] = None

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def merge_and_save(adapter_path, output_dir):
    print(f"Executing deep tensor fusion for {adapter_path}...")
    base = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cpu"
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    merged_model = model.merge_and_unload()

    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Successfully compiled integrated model structure to {output_dir}\\n")

merge_and_save("./medical_lora_adapter", "./merged_lora")
merge_and_save("./medical_mtlora_adapter", "./merged_mtlora")
''')

print("\nRunning clean sub-process model fusion...")
!python merge_models.py

# ========================================================
# 3. CONVERT COMPLETED HF DIRECTORIES TO BASE GGUF
# ========================================================
print("\nConverting directories to GGUF format...")
!python llama.cpp/convert_hf_to_gguf.py ./merged_lora --outfile ./lora_fp16.gguf --outtype f16
!python llama.cpp/convert_hf_to_gguf.py ./merged_mtlora --outfile ./mtlora_fp16.gguf --outtype f16

# ========================================================
# 4. EXPORT TO 4-BIT QUANTIZATION
# ========================================================
print("\nQuantizing models to 4-bit (Q4_K_M) for Raspberry Pi...")
# Ensure quantize binary is built if it isn't already
!cd llama.cpp && cmake . -B build && cmake --build build --config Release
!./llama.cpp/build/bin/llama-quantize ./lora_fp16.gguf ./medical_lora_q4.gguf Q4_K_M
!./llama.cpp/build/bin/llama-quantize ./mtlora_fp16.gguf ./medical_mtlora_q4.gguf Q4_K_M

# Verify files exist before running the download hook
import os
if os.path.exists('medical_lora_q4.gguf') and os.path.exists('medical_mtlora_q4.gguf'):
    print("\nSuccess! Both models are successfully built into Q4_K_M GGUF format.")
    print("Initiating browser file downloads...")
    from google.colab import files
    files.download('medical_lora_q4.gguf')
    files.download('medical_mtlora_q4.gguf')
else:
    print("\n[ERROR]: GGUF files were not created. Check terminal output log traces above.")

Setting up llama.cpp tools...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 103910, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 103910 (delta 87), reused 41 (delta 41), pack-reused 103740 (from 3)
Receiving objects: 100% (103910/103910), 409.67 MiB | 29.38 MiB/s, done.
Resolving deltas: 100% (72628/72628), done.

Extracting uploaded adapters...
Archive:  medical_adapters.zip
  inflating: medical_lora_adapter/README.md  
  inflating: medical_lora_adapter/adapter_model.safetensors  
  inflating: medical_lora_adapter/adapter_config.json  
  inflating: medical_mtlora_adapter/README.md  
  inflating: medical_mtlora_adapter/adapter_model.safetensors  
  inflating: medical_mtlora_adapter/adapter_config.json  

Running clean sub-process model fusion...
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/tor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Launch the llama.cpp web server on a fresh port (8090)
!./llama.cpp/build/bin/llama-server -m ./medical_lora_q4.gguf -c 2048 --port 8090

/bin/bash: line 1: ./llama.cpp/build/bin/llama-server: No such file or directory


In [ ]:
from google.colab.output import eval_js
link = eval_js('google.colab.kernel.proxyPort(8090)')
print("-" * 50)
print(f"CLICK HERE TO CHAT: {link}")
print("-" * 50)